# Validate time-slice GDS projections

This notebook proves that a graph created with `gds.graph.filter` contains the same transaction relationships as a naïve Cypher projection for one time window. It uses a small, disposable transaction dataset.

**Requirements:** a Neo4j DBMS with the GDS plugin, plus credentials in `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, and optionally `NEO4J_DB`. It requires GDS 2.5 or later.

In [9]:
%pip install -q graphdatascience pandas python-dotenv

import os
from dotenv import find_dotenv, load_dotenv
from time import perf_counter

import pandas as pd
from pandas.testing import assert_frame_equal
from graphdatascience import GraphDataScience, ServerVersion

dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path, override=False)

NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_DB = os.environ.get('NEO4J_DB', 'neo4j')
NEO4J_AUTH = (
    os.environ.get('NEO4J_USERNAME', 'neo4j'),
    os.environ.get('NEO4J_PASSWORD', ''),
)

gds = GraphDataScience(NEO4J_URI, auth=NEO4J_AUTH, database=NEO4J_DB)
print('GDS version:', gds.server_version())
assert gds.server_version() >= ServerVersion(2, 5, 0), 'Use GDS 2.5 or later.'


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
GDS version: 2026.10.1


## Create a disposable source graph

The setup cell deletes only nodes carrying the `TimeSliceDemo` label. It creates a realistic source shape—transaction nodes between accounts—then the projections will synthesize account-to-account `TRANSFER` relationships. Times are Unix epoch seconds.

In [10]:
account_count = 1_000
transaction_count = 100_000
batch_size = 1_000
start_epoch = 1736035200
spacing_seconds = 60

gds.run_cypher("MATCH (n:TimeSliceDemo) DETACH DELETE n")

# 10k accounts is safe to create in one query.
gds.run_cypher("""
    UNWIND range(1, $account_count) AS n
    CREATE (:Account:TimeSliceDemo {
      accountId: 'A-' + right('000000' + toString(n), 6)
    })
""", params={"account_count": account_count})

# Each call is its own transaction, avoiding CALL ... IN TRANSACTIONS.
transaction_query = """
    UNWIND range($batch_start, $batch_end) AS i
    WITH i,
         toInteger(rand() * $account_count) + 1 AS source_n,
         toInteger(rand() * $account_count) + 1 AS random_target_n,
         rand() AS target_roll,
         toFloat(1000 + toInteger(rand() * 99001)) / 100.0 AS amount,
         toInteger(rand() * 86400) AS time_jitter

    WITH i, source_n, amount, time_jitter,
         CASE
           WHEN target_roll < 0.35
             THEN toInteger(rand() * 100) + 1
           ELSE random_target_n
         END AS candidate_target_n

    WITH i, source_n, amount, time_jitter,
         CASE
           WHEN candidate_target_n = source_n
             THEN (source_n % $account_count) + 1
           ELSE candidate_target_n
         END AS target_n

    MATCH (source:Account:TimeSliceDemo {
      accountId: 'A-' + right('000000' + toString(source_n), 6)
    })
    MATCH (target:Account:TimeSliceDemo {
      accountId: 'A-' + right('000000' + toString(target_n), 6)
    })

    CREATE (tx:Transaction:TimeSliceDemo {
      transactionId: 'T-' + right('0000000' + toString(i + 1), 7),
      occurredAt: $start_epoch + (i * $spacing_seconds) + time_jitter,
      amount: amount
    })
    CREATE (source)-[:SENT {weight: amount}]->(tx)
    CREATE (tx)-[:RECEIVED_BY {weight: amount}]->(target)
"""

for batch_start in range(0, transaction_count, batch_size):
    batch_end = min(batch_start + batch_size - 1, transaction_count - 1)

    gds.run_cypher(
        transaction_query,
        params={
            "batch_start": batch_start,
            "batch_end": batch_end,
            "account_count": account_count,
            "start_epoch": start_epoch,
            "spacing_seconds": spacing_seconds,
        },
    )

gds.run_cypher("""
    MATCH (a:Account:TimeSliceDemo)
    WITH count(a) AS accounts
    MATCH (tx:Transaction:TimeSliceDemo)
    RETURN accounts, count(tx) AS transactions,
           min(tx.occurredAt) AS firstOccurredAt,
           max(tx.occurredAt) AS lastOccurredAt
""")

[#DC54]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))): OSError('No data')


KeyboardInterrupt: 

In [ ]:
BASE_GRAPH = 'demo_transfers_all_time'
SLICE_GRAPH = 'demo_transfers_january'
NAIVE_GRAPH = 'demo_transfers_january_naive'

# Re-running this notebook is safe: remove only its three catalog graphs.
for graph_name in (BASE_GRAPH, SLICE_GRAPH, NAIVE_GRAPH):
    if gds.graph.exists(graph_name):
        G = gds.graph.get(graph_name)
        gds.graph.drop(G)

# Half-open window: 2025-01-10T00:00:00Z through (not including) 2025-02-01T00:00:00Z.
start_at = 1736467200
end_at = 1738368000

## Reference implementation: project the requested time window

This is intentionally the approach we want to avoid in a highly interactive application. It is our correctness baseline.

In [ ]:
NAIVE_PROJECTION = '''
MATCH (source:Account:TimeSliceDemo)-[:SENT]->(tx:Transaction:TimeSliceDemo)-[:RECEIVED_BY]->(target:Account:TimeSliceDemo)
WHERE tx.occurredAt >= $start_at AND tx.occurredAt < $end_at
RETURN gds.graph.project(
  $graph_name, source, target,
  {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: 'TRANSFER',
    relationshipProperties: {occurredAt: tx.occurredAt, amount: tx.amount}
  }
)
'''

started = perf_counter()
naive_graph, naive_result = gds.graph.project.cypher(
    query=NAIVE_PROJECTION, graph_name=NAIVE_GRAPH, start_at=start_at, end_at=end_at
)
naive_elapsed_ms = (perf_counter() - started) * 1_000
print(f'Naïve projection: {naive_graph.relationship_count()} relationships; {naive_elapsed_ms:.1f} ms client wall time')

## Optimized implementation: project once, then filter in the graph catalog

The broad projection retains `occurredAt` and `amount`. The `parameters` map is important: it passes safe values into the GDS relationship predicate rather than interpolating a user-selected date into the predicate string.

In [ ]:
BASE_PROJECTION = '''
MATCH (source:Account:TimeSliceDemo)-[:SENT]->(tx:Transaction:TimeSliceDemo)-[:RECEIVED_BY]->(target:Account:TimeSliceDemo)
RETURN gds.graph.project(
  $graph_name, source, target,
  {
    sourceNodeLabels: labels(source),
    targetNodeLabels: labels(target),
    relationshipType: 'TRANSFER',
    relationshipProperties: {occurredAt: tx.occurredAt, amount: tx.amount}
  }
)
'''

started = perf_counter()
base_graph, base_result = gds.graph.project.cypher(query=BASE_PROJECTION, graph_name=BASE_GRAPH)
base_elapsed_ms = (perf_counter() - started) * 1_000

started = perf_counter()
G = gds.graph.get(BASE_GRAPH)
filter_result = gds.graph.filter(
    G,
    SLICE_GRAPH,
    '*',
    'r.occurredAt >= $startAt AND r.occurredAt < $endAt',
    {'start_at': start_at, 'end_at': end_at}
)

filter_elapsed_ms = (perf_counter() - started) * 1_000
filtered_graph = gds.graph.get(SLICE_GRAPH)
print(f'Broad projection: {base_graph.relationship_count()} relationships; {base_elapsed_ms:.1f} ms client wall time')
print(f'Filtered projection: {filtered_graph.relationship_count()} relationships; {filter_elapsed_ms:.1f} ms client wall time')

## Correctness check

We compare three sources of truth: the database time predicate, the naïve projection, and the filtered projection. The assertion is the proof; it fails if the relationships selected by either projection differ.

In [ ]:
def projected_edges(graph_name):
    return gds.run_cypher(
        '''
        CALL gds.graph.relationshipProperty.stream($graph_name, 'occurredAt')
        YIELD sourceNodeId, targetNodeId, propertyValue
        RETURN gds.util.asNode(sourceNodeId).accountId AS source,
               gds.util.asNode(targetNodeId).accountId AS target,
               propertyValue AS occurredAt
        ORDER BY occurredAt, source, target
        ''',
        params={'graph_name': graph_name},
    ).reset_index(drop=True)

expected = gds.run_cypher(
    '''
    MATCH (source:Account:TimeSliceDemo)-[:SENT]->(tx:Transaction:TimeSliceDemo)-[:RECEIVED_BY]->(target:Account:TimeSliceDemo)
    WHERE tx.occurredAt >= $start_at AND tx.occurredAt < $end_at
    RETURN source.accountId AS source, target.accountId AS target, tx.occurredAt AS occurredAt
    ORDER BY occurredAt, source, target
    ''',
    params={'start_at': start_at, 'end_at': end_at},
).reset_index(drop=True)

naive_edges = projected_edges(NAIVE_GRAPH)
filtered_edges = projected_edges(SLICE_GRAPH)
assert_frame_equal(expected, naive_edges, check_dtype=False)
assert_frame_equal(expected, filtered_edges, check_dtype=False)
assert naive_graph.relationship_count() == filtered_graph.relationship_count() == len(expected)
print('PASS: database, naïve projection, and filtered projection selected the same relationships.')
expected

## Run an algorithm on the time slice

`nodeFilter: '*'` preserves accounts from the broad graph even when they have no selected transfer. PageRank therefore includes those isolated accounts with a zero (or near-zero) score. Choose a different node filter only when that is the intended analytical population.

In [ ]:
page_rank = gds.run_cypher(
    '''
    CALL gds.pageRank.stream($graph_name, {relationshipWeightProperty: 'amount'})
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).accountId AS accountId, score
    ORDER BY score DESC, accountId
    ''',
    params={'graph_name': SLICE_GRAPH},
)
page_rank

## Cleanup

Run this when finished. It drops only this notebook's catalog graphs and deletes only the labelled demo data. In an application, drop the temporary slice after each request; keep and refresh the broad projection according to your data-freshness policy.

In [ ]:
for graph_name in (NAIVE_GRAPH, SLICE_GRAPH, BASE_GRAPH):
    if gds.graph.exists(graph_name):
        G = gds.graph.get(graph_name)
        gds.graph.drop(G)
gds.run_cypher('MATCH (n:TimeSliceDemo) DETACH DELETE n')
gds.close()
print('Demo graphs and demo data removed.')